In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("CyberBullying_tweets.csv")

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11100 entries, 0 to 11099
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Text      11100 non-null  object
 1   CB_Label  11100 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 173.6+ KB


In [4]:
data.head()

,Text,CB_Label
0,damn there is someones nana up here at beach w...,0
1,no kidding! dick clark was a corpse mechanical...,0
2,i read an article on jobros and thought damn w...,0
3,I got one fucking day of sprinkles and now it'...,0
4,I was already listening to Elliott smith and ...,0


In [5]:
data.shape

(11100, 2)

In [6]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print(data.head())


                                                Text  CB_Label
0  damn there is someones nana up here at beach w...         0
1  no kidding! dick clark was a corpse mechanical...         0
2  i read an article on jobros and thought damn w...         0
3  I got one fucking day of sprinkles and now it'...         0
4  I was already listening to Elliott smith  and ...         0


In [7]:
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources
nltk.download('punkt')

# Function for text cleaning
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#\w+', '', text)  # Remove hashtags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

# Apply cleaning to the 'Text' column
data['clean_text'] = data['Text'].apply(clean_text)
data.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


,Text,CB_Label,clean_text
0,damn there is someones nana up here at beach w...,0,damn there is someones nana up here at beach w...
1,no kidding! dick clark was a corpse mechanical...,0,no kidding dick clark was a corpse mechanicall...
2,i read an article on jobros and thought damn w...,0,i read an article on jobros and thought damn w...
3,I got one fucking day of sprinkles and now it'...,0,i got one fucking day of sprinkles and now its...
4,I was already listening to Elliott smith and ...,0,i was already listening to elliott smith and i...


In [8]:
from nltk.tokenize import RegexpTokenizer

# Initialize the tokenizer (splits text into words)
tokenizer = RegexpTokenizer(r'\w+')

# Tokenize the 'clean_text' column and create a new 'tokens' column
data['tokens'] = data['clean_text'].apply(tokenizer.tokenize)

# Print the DataFrame to see the tokenized text
print(data[['clean_text', 'tokens']].head())
data.head()

                                          clean_text  \
0  damn there is someones nana up here at beach w...   
1  no kidding dick clark was a corpse mechanicall...   
2  i read an article on jobros and thought damn w...   
3  i got one fucking day of sprinkles and now its...   
4  i was already listening to elliott smith and i...   

                                              tokens  
0  [damn, there, is, someones, nana, up, here, at...  
1  [no, kidding, dick, clark, was, a, corpse, mec...  
2  [i, read, an, article, on, jobros, and, though...  
3  [i, got, one, fucking, day, of, sprinkles, and...  
4  [i, was, already, listening, to, elliott, smit...  


,Text,CB_Label,clean_text,tokens
0,damn there is someones nana up here at beach w...,0,damn there is someones nana up here at beach w...,"[damn, there, is, someones, nana, up, here, at..."
1,no kidding! dick clark was a corpse mechanical...,0,no kidding dick clark was a corpse mechanicall...,"[no, kidding, dick, clark, was, a, corpse, mec..."
2,i read an article on jobros and thought damn w...,0,i read an article on jobros and thought damn w...,"[i, read, an, article, on, jobros, and, though..."
3,I got one fucking day of sprinkles and now it'...,0,i got one fucking day of sprinkles and now its...,"[i, got, one, fucking, day, of, sprinkles, and..."
4,I was already listening to Elliott smith and ...,0,i was already listening to elliott smith and i...,"[i, was, already, listening, to, elliott, smit..."


In [9]:
data.head(10)

,Text,CB_Label,clean_text,tokens
0,damn there is someones nana up here at beach w...,0,damn there is someones nana up here at beach w...,"[damn, there, is, someones, nana, up, here, at..."
1,no kidding! dick clark was a corpse mechanical...,0,no kidding dick clark was a corpse mechanicall...,"[no, kidding, dick, clark, was, a, corpse, mec..."
2,i read an article on jobros and thought damn w...,0,i read an article on jobros and thought damn w...,"[i, read, an, article, on, jobros, and, though..."
3,I got one fucking day of sprinkles and now it'...,0,i got one fucking day of sprinkles and now its...,"[i, got, one, fucking, day, of, sprinkles, and..."
4,I was already listening to Elliott smith and ...,0,i was already listening to elliott smith and i...,"[i, was, already, listening, to, elliott, smit..."
5,I told Derek to go fuck himself Devyn told us...,0,i told derek to go fuck himself devyn told us ...,"[i, told, derek, to, go, fuck, himself, devyn,..."
6,I'm watching the new Smosh video and laughing ...,0,im watching the new smosh video and laughing m...,"[im, watching, the, new, smosh, video, and, la..."
7,My mom didn't like Catholicism because the ide...,0,my mom didnt like catholicism because the idea...,"[my, mom, didnt, like, catholicism, because, t..."
8,ya i know! lol AND big thunder mountain broke ...,0,ya i know lol and big thunder mountain broke d...,"[ya, i, know, lol, and, big, thunder, mountain..."
9,"lucky bitch.....""",0,lucky bitch,"[lucky, bitch]"


In [10]:
import torch
from transformers import BertTokenizer, BertModel
import pandas as pd

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

def get_bert_embeddings(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

data["bert_embeddings"] = data["clean_text"].apply(lambda x: get_bert_embeddings([x])[0])




/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [11]:
import torch
from transformers import BertTokenizer, BertModel
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

X = np.vstack(data["bert_embeddings"].values)
y = data["CB_Label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

logistic_model = LogisticRegression(max_iter=500)
logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", report)


Accuracy: 0.7563
Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.78      0.76      1110
           1       0.77      0.73      0.75      1110

    accuracy                           0.76      2220
   macro avg       0.76      0.76      0.76      2220
weighted avg       0.76      0.76      0.76      2220



In [15]:
!pip install transformers datasets
import torch
from transformers import BertTokenizer, BertModel
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from datasets import load_dataset


# Load the dataset using 'datasets' library
dataset = load_dataset("csv", data_files="CyberBullying_tweets.csv")
# Convert the dataset to a Pandas DataFrame
data = dataset["train"].to_pandas()


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")


def get_bert_embeddings(text):
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=512)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).numpy()

# Generate BERT embeddings for the 'clean_text' column and add them to the DataFrame
# This line was missing, causing the KeyError
data["bert_embeddings"] = data["Text"].apply(lambda x: get_bert_embeddings([x])[0])

# Rest of your code remains the same...
X = np.vstack(data["bert_embeddings"].values)
y = data["CB_Label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


logistic_model = LogisticRegression(max_iter=500)
logistic_model.fit(X_train, y_train)


y_pred = logistic_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", report)


new_text = input("Enter a text: ")
new_embedding = get_bert_embeddings([new_text])[0]
prediction = logistic_model.predict([new_embedding])[0]

label = "Cyberbullying" if prediction == 1 else "Not Cyberbullying"
print(f"Prediction: {label}")

Accuracy: 0.7698
Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.83      0.78      1110
           1       0.81      0.71      0.75      1110

    accuracy                           0.77      2220
   macro avg       0.77      0.77      0.77      2220
weighted avg       0.77      0.77      0.77      2220

Enter a text: Hey Idiot shut up
Prediction: Cyberbullying
